### 07 - Creating python function to integrate with Corpus Notebook

In this notebook, we will create a function that can take CG3-formatted text and convert it into a list‑of‑lists‑of‑tuples structure, corresponding to a list of sentences, a list of words for every sentence, and a tuple of the word and disambiguated FST readings for each word.

In [ ]:
from cg3_process import disambiguate
from fst_runtime.fst import Fst

# test parsing multiple lines
ojibwe_text = """
Ningoding zaaga’igan omadaagon*.

Ezhi-maajiiyaadagaagod, wabigamaanig beshwaabandang, awiya owaabamaan bimaadagaagopatoonid; aazha miinawaa, niiwiwa’.

Goniginiin, mahiingana’!

Ezhi-biibaagimaad: “Nijiimijaa* [nishiimisaa] akawe, ga-waabamininim!”

Geget gii-bijibatoowa’; ezhiwawenabinid ani-naazikawaad.

Ezhi-ganoonaad: “Niiji-saziikizi, aandi ezhaayeg?”

“Kaa, o’owidi, giizhigadikwaning, mii iwidi ezhaayaang.

Niibinong gii-asanjigoobanig ogow gidooshimag, gichi-ayaaben ogii-nisaawaabaniin.

Mii dash iwidi ezhaayaang.

“Ediwe, mii gaye niin iwidi ezhaayaan, giizhigadikwaning, - mii sa i’iw ji-ani-waawiijiiwinagog.

Aaniish, mii iw zhigwa w[e]naagoshininig.

“Aaniish i’iw, Jiijiigwaanoowis,(1) ani-nanda-oninamaasiwan, maagizhaa da-kisinaa dibikad.

Daga, gi-mishoome’iwaa dani-andoo-oninamaa.”

Aaniish, mii sa geget Nenabosho ani-nanda’oninamaad.
"""

GRAMAMR_FILE = "/Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/Ojibwe_Constraint_Grammar/data/CG3_rules/Ojibwe_disambiguation.cg3"
FST_FILE = "/Users/matthias/ELF-Lab Repos/Ojibwe_Constraint_Grammar/Ojibwe_Constraint_Grammar/data/fst/ojibwe.att"
fst_obj = Fst(FST_FILE)

result = disambiguate(sentence=ojibwe_text, cg3_grammar_filepath=GRAMAMR_FILE, fst=fst_obj, verbose=True)
result

Before parsing:
"<ningoding>"
	"ningoding" ADVTmp
"<zaaga’igan>"

"<omadaagon*>"

"<.>"

"<ezhi-maajiiyaadagaagod>"

"<,>"

"<wabigamaanig>"
	"wabigamaa" VII Cnj Pos Neu 0PlObvSubj
	"wabigamaa" VII Cnj Pos Neu 0SgObvSubj
"<beshwaabandang>"
	"beshwaabandan" VTI Cnj Pos Neu 3SgProxSubj 0SgObj
	"beshwaabandan" VTI Cnj Pos Neu 3SgProxSubj 0PlObj
"<,>"

"<awiya>"
	"awiya" PRONIndf
"<owaabamaan>"
	"waabam" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj
	"waabam" VTA Ind Pos Neu 3SgProxSubj 3PlObvObj
"<bimaadagaagopatoonid;>"

"<aazha>"
	"aazha" ADVTmp
"<miinawaa>"
	"miinawaa" ADVConj
"<,>"

"<niiwiwa’>"

"<.>"

"<goniginiin>"

"<,>"

"<mahiingana’>"

"<!>"

"<ezhi-biibaagimaad:>"

"<“nijiimijaa*>"

"<[nishiimisaa]>"

"<akawe>"
	"akawe" ADVTmp
"<,>"

"<ga-waabamininim!”>"

"<geget>"
	"geget" ADVMan
"<gii-bijibatoowa’;>"

"<ezhiwawenabinid>"

"<ani-naazikawaad>"
	"naazikaw" PVDir/ani VTA Cnj Pos Neu 3SgProxSubj 3SgObvObj
	"naazikaw" PVDir/ani VTA Cnj Pos Neu 3SgProxSubj 3PlObvObj
"<.>"

"<ezhi-ganoonaa

'"<ningoding>"\n\t"ningoding" ADVTmp\n"<zaaga’igan>"\n"<omadaagon*>"\n"<.>"\n"<ezhi-maajiiyaadagaagod>"\n"<,>"\n"<wabigamaanig>"\n\t"wabigamaa" VII Cnj Pos Neu 0PlObvSubj\n\t"wabigamaa" VII Cnj Pos Neu 0SgObvSubj\n"<beshwaabandang>"\n\t"beshwaabandan" VTI Cnj Pos Neu 3SgProxSubj 0SgObj\n"<,>"\n"<awiya>"\n\t"awiya" PRONIndf\n"<owaabamaan>"\n\t"waabam" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj\n"<bimaadagaagopatoonid;>"\n"<aazha>"\n\t"aazha" ADVTmp\n"<miinawaa>"\n\t"miinawaa" ADVConj\n"<,>"\n"<niiwiwa’>"\n"<.>"\n"<goniginiin>"\n"<,>"\n"<mahiingana’>"\n"<!>"\n"<ezhi-biibaagimaad:>"\n"<“nijiimijaa*>"\n"<[nishiimisaa]>"\n"<akawe>"\n\t"akawe" ADVTmp\n"<,>"\n"<ga-waabamininim!”>"\n"<geget>"\n\t"geget" ADVMan\n"<gii-bijibatoowa’;>"\n"<ezhiwawenabinid>"\n"<ani-naazikawaad>"\n\t"naazikaw" PVDir/ani VTA Cnj Pos Neu 3SgProxSubj 3SgObvObj\n"<.>"\n"<ezhi-ganoonaad:>"\n"<“niiji-saziikizi>"\n"<,>"\n"<aandi>"\n\t"aandi" ADVInter\n"<ezhaayeg?”>"\n"<“kaa>"\n"<,>"\n"<o’owidi>"\n"<,>"\n"<giizhigadikwaning>"\n"

In [2]:
import re
from typing import List, Tuple

def cg3_to_sentences(cg3_text: str) -> List[List[Tuple[str, List[str]]]]:
    """
    Convert CG3‑formatted text into a list‑of‑lists‑of‑tuples structure.

    Parameters
    ----------
    cg3_text : str
        The raw CG3 output, e.g.:
        "<ningoding>"
            "ningoding" ADVTmp
        "<zaaga’igan>"
        "<omadaagon*>"
        "<.>"

    Returns
    -------
    List[List[Tuple[str, List[str]]]]
        * Outer list  : sentences
        * Inner list  : (word, analyses) pairs
        * Tuple[0]    : the token surface form as a string
        * Tuple[1]    : list of analysis strings for that token
    """
    # regex helpers
    tok_line_re     = re.compile(r'^\s*"<([^>]+)>"\s*$')
    analysis_line_re = re.compile(r'^\s*"(.+?)"\s*(.*)$')

    sentences: List[List[Tuple[str, List[str]]]] = []
    cur_sentence: List[Tuple[str, List[str]]] = []
    cur_word   : str | None   = None
    cur_readings: List[str]   = []

    for raw in cg3_text.splitlines():
        line = raw.rstrip()

        # Sentence boundary: completely blank line
        if not line.strip():
            if cur_word is not None:
                cur_sentence.append((cur_word, cur_readings))
                cur_word, cur_readings = None, []
            if cur_sentence:
                sentences.append(cur_sentence)
                cur_sentence = []
            continue

        # New token line ("<word>") 
        tok_match = tok_line_re.match(line)
        if tok_match:
            # flush previous token
            if cur_word is not None:
                cur_sentence.append((cur_word, cur_readings))
            cur_word     = tok_match.group(1)  # actual surface form
            cur_readings = []
            continue

        # Analysis line (tab‑indented) 
        if cur_word is not None and line.lstrip().startswith('"'):
            cur_readings.append(line.strip())
            continue


    # Flush last token / sentence 
    if cur_word is not None:
        cur_sentence.append((cur_word, cur_readings))
    if cur_sentence:
        sentences.append(cur_sentence)

    return sentences


In [3]:
parsed = cg3_to_sentences(result)
print(parsed)
print(type(parsed))

[[('ningoding', ['"ningoding" ADVTmp']), ('zaaga’igan', []), ('omadaagon*', []), ('.', []), ('ezhi-maajiiyaadagaagod', []), (',', []), ('wabigamaanig', ['"wabigamaa" VII Cnj Pos Neu 0PlObvSubj', '"wabigamaa" VII Cnj Pos Neu 0SgObvSubj']), ('beshwaabandang', ['"beshwaabandan" VTI Cnj Pos Neu 3SgProxSubj 0SgObj']), (',', []), ('awiya', ['"awiya" PRONIndf']), ('owaabamaan', ['"waabam" VTA Ind Pos Neu 3SgProxSubj 3SgObvObj']), ('bimaadagaagopatoonid;', []), ('aazha', ['"aazha" ADVTmp']), ('miinawaa', ['"miinawaa" ADVConj']), (',', []), ('niiwiwa’', []), ('.', []), ('goniginiin', []), (',', []), ('mahiingana’', []), ('!', []), ('ezhi-biibaagimaad:', []), ('“nijiimijaa*', []), ('[nishiimisaa]', []), ('akawe', ['"akawe" ADVTmp']), (',', []), ('ga-waabamininim!”', []), ('geget', ['"geget" ADVMan']), ('gii-bijibatoowa’;', []), ('ezhiwawenabinid', []), ('ani-naazikawaad', ['"naazikaw" PVDir/ani VTA Cnj Pos Neu 3SgProxSubj 3SgObvObj']), ('.', []), ('ezhi-ganoonaad:', []), ('“niiji-saziikizi', [])